# 01 - Exploratory Data Analysis (NSL-KDD)
Goal: understand the data before building anything.

In [1]:
import sys; sys.path.append("..")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.features import COLS, FEATURES
train_df = pd.read_csv("../data/raw/KDDTrain+.txt", names=COLS)
test_df = pd.read_csv("../data/raw/KDDTest+.txt", names=COLS)
print(train_df.shape, test_df.shape)   # expect (125973, 43) and (22544, 43)
train_df.head()

(125973, 43) (22544, 43)


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


## Class balance (0 = normal, 1 = attack)

In [2]:
train_df["is_attack"] = (train_df["label"] != "normal").astype(int)
test_df["is_attack"] = (test_df["label"] != "normal").astype(int)
print("TRAIN\n", train_df["is_attack"].value_counts())
print("TEST\n", test_df["is_attack"].value_counts())
train_df["label"].value_counts().head(10)

TRAIN
 is_attack
0    67343
1    58630
Name: count, dtype: int64
TEST
 is_attack
1    12833
0     9711
Name: count, dtype: int64


label
normal         67343
neptune        41214
satan           3633
ipsweep         3599
portsweep       2931
smurf           2646
nmap            1493
back             956
teardrop         892
warezclient      890
Name: count, dtype: int64

## Attacks that exist ONLY in the test file
The autoencoder never sees any attack in training, but these are completely new even for supervised models.

In [3]:
only_test = sorted(set(test_df["label"]) - set(train_df["label"]))
print(len(only_test), "attack types only in test:", only_test)

17 attack types only in test: ['apache2', 'httptunnel', 'mailbomb', 'mscan', 'named', 'processtable', 'ps', 'saint', 'sendmail', 'snmpgetattack', 'snmpguess', 'sqlattack', 'udpstorm', 'worm', 'xlock', 'xsnoop', 'xterm']


## Categorical columns (need one-hot encoding)

In [4]:
for col in ["protocol_type", "service", "flag"]:
    print(col, "-> train:", train_df[col].nunique(), "| test:", test_df[col].nunique())
print("service values in only one file:", set(train_df["service"]) ^ set(test_df["service"]))

protocol_type -> train: 3 | test: 3
service -> train: 70 | test: 64
flag -> train: 11 | test: 11
service values in only one file: {'urh_i', 'aol', 'http_8001', 'harvest', 'http_2784', 'red_i'}


## Constant columns (carry no information)

In [5]:
n_unique = train_df[FEATURES].nunique()
print(n_unique[n_unique == 1])

num_outbound_cmds    1
dtype: int64


## Value ranges and missing values

In [6]:
print("Missing values:", train_df[FEATURES].isnull().sum().sum())
stats = train_df[FEATURES].describe().T[["min", "max", "mean"]]
stats.sort_values("max", ascending=False).head(8)

Missing values: 0


,min,max,mean
src_bytes,0.0,1.379964e+09,45566.743000
dst_bytes,0.0,1.309937e+09,19779.114421
duration,0.0,4.290800e+04,287.144650
num_compromised,0.0,7.479000e+03,0.279250
num_root,0.0,7.468000e+03,0.302192
count,0.0,5.110000e+02,84.107555
srv_count,0.0,5.110000e+02,27.737888
dst_host_srv_count,0.0,2.550000e+02,115.653005


## Normal vs attack: one feature (log scale)

In [7]:
plt.figure(figsize=(8, 4))
for v, name in [(0, "normal"), (1, "attack")]:
    s = train_df[train_df["is_attack"] == v]
    plt.hist(np.log1p(s["src_bytes"]), bins=50, alpha=0.5, label=name)
plt.xlabel("log(1 + src_bytes)"); plt.ylabel("flows"); plt.legend(); plt.show()